# Regular Expressions

> Generated from [`07-Regular-Expressions.py`](07-Regular-Expressions.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Using the `re` module to search, match, extract, and replace text with patterns — from basic matching through groups, lookaheads, and performance pitfalls.

### Key points
- `re.match(pattern, s)` matches only at the **start** of the string; `re.search(pattern, s)` finds a match **anywhere**.
- `re.findall(pattern, s)` returns all non-overlapping matches as a list; `re.finditer()` gives an iterator of match objects (with `.group()` and `.span()` for position).
- `re.sub(pattern, replacement, s)` finds and replaces; `re.split(pattern, s)` splits a string on a pattern.
- Common building blocks: `\d` digit, `\w` word char, `\s` whitespace, `.` any char, `+` one-or-more, `*` zero-or-more, `?` optional, `{n,m}` range, `^`/`$` start/end of string.
- Groups: `(...)` captures part of a match, accessed via `match.group(1)`, `.group(2)`, etc. Named groups `(?P<name>...)` are accessed via `match.groupdict()` or `.group("name")`.
- `re.compile(pattern)` precompiles a pattern for efficient reuse when matching many strings against the same pattern.
- Lookaheads/lookbehinds check for a pattern **without** including it in the match: `(?=...)` positive lookahead, `(?!...)` negative lookahead, `(?<=...)` positive lookbehind, `(?<!...)` negative lookbehind.
- Backreferences (`\1`, `\2`, ...) match text a prior group already captured — useful for catching repeated words. Named version: `(?P=name)`.
- `re.VERBOSE` (`re.X`) lets you spread a pattern across multiple lines with comments and ignored whitespace, for readability.

### Gotchas (the part worth revising twice)
- **Greedy vs non-greedy**: quantifiers (`*`, `+`, `?`, `{n,m}`) are greedy by default — they match as much as possible (`<.*>` on `<b>x</b>` spans to the LAST `>`). Add `?` after them (`<.*?>`) to make them non-greedy/lazy — matches as little as possible, stopping at the first `>`.
- **Catastrophic backtracking**: nested quantifiers like `(a+)+` can force the regex engine to try an exponential number of ways to split the input before giving up on a failed match — a pattern like `(a+)+b` against a long run of `a`s with no trailing `b` can hang a program for minutes. Avoid nested/ambiguous quantifiers; prefer a flatter equivalent like `a+b`.
- Forgetting `encoding`/raw strings (`r"..."`) is a common source of bugs — always use raw strings for regex patterns so backslashes aren't interpreted by Python first.

### When it matters
Validation (emails, formats), extracting structured data from unstructured text, and search/replace across text — but reach for a real parser instead of regex once the structure gets nested or complex (e.g. don't parse HTML/JSON with regex).

### One-line recap example
```python
import re
re.findall(r"\d+", "I have 3 cats and 12 fish")   # ['3', '12']
```

In [ ]:
import re

## 01. Regular Expressions - Introduction

- The `re` module lets you search, match, and manipulate strings using
  patterns (regex) - useful for validation, extraction, and text processing.

## 02. re.match() - Match at the Start of a String

In [ ]:
result = re.match(r"Hello", "Hello, World!")
print("\n# 02. re.match()")
print(result)
print("matched text:", result.group() if result else None)

## 03. re.search() - Search Anywhere in a String

In [ ]:
result = re.search(r"World", "Hello, World!")
print("\n# 03. re.search()")
print("found:", result.group() if result else None)

## 04. re.findall() - Find All Non-Overlapping Matches

In [ ]:
text = "My numbers are 42, 17, and 8."
numbers = re.findall(r"\d+", text)
print("\n# 04. re.findall()")
print(numbers)

## 05. re.finditer() - Iterator of Match Objects (with positions)

In [ ]:
for match in re.finditer(r"\d+", text):
    print(f"matched '{match.group()}' at position {match.span()}")

## 06. re.sub() - Search and Replace

In [ ]:
censored = re.sub(r"\d+", "#", text)
print("\n# 06. re.sub()")
print(censored)

## 07. Common Pattern Building Blocks

In [ ]:
print(re.findall(r"\w+", "Python is fun!"))       # words
print(re.findall(r"^\w+", "Python is fun!"))       # first word only
print(re.findall(r"\w+$", "Python is fun"))         # last word only (no trailing !)

## 08. Groups - Capturing Parts of a Match

In [ ]:
match = re.match(r"(\w+)@(\w+)\.com", "user@example.com")
print("\n# 08. Groups")
if match:
    print("full match:", match.group(0))
    print("username:", match.group(1))
    print("domain:", match.group(2))

## 09. Named Groups

In [ ]:
match = re.match(r"(?P<username>\w+)@(?P<domain>\w+)\.com", "admin@python.com")
print("\n# 09. Named Groups")
if match:
    print(match.groupdict())

## 10. Compiling Patterns for Reuse

- re.compile() precompiles a pattern for efficient repeated use.

In [ ]:
email_pattern = re.compile(r"^[\w.\-]+@[\w\-]+\.\w+$")
print("\n# 10. Compiling Patterns for Reuse")
for email in ["good@example.com", "not-an-email", "also.good@sub.example.com"]:
    is_valid = bool(email_pattern.match(email))
    print(f"{email}: {'valid' if is_valid else 'invalid'}")

## 11. Splitting Strings with re.split()

In [ ]:
parts = re.split(r"[,;]\s*", "apple, banana; cherry,  date")
print("\n# 11. re.split()")
print(parts)

## 12. Greedy vs Non-Greedy Quantifiers

- By default, quantifiers (*, +, ?, {n,m}) are GREEDY - they match as
  much text as possible. Adding a `?` after them makes them NON-GREEDY
  (a.k.a. lazy) - they match as LITTLE text as possible.

In [ ]:
html_like = "<b>bold</b> and <i>italic</i>"
print("\n# 12. Greedy vs Non-Greedy Quantifiers")
print("greedy    .* :", re.findall(r"<.*>", html_like))    # spans from first < to LAST >
print("non-greedy .*?:", re.findall(r"<.*?>", html_like))  # stops at the FIRST >

## 13. Lookahead and Lookbehind Assertions

- These check for a pattern WITHOUT including it in the match.
- (?=...)  positive lookahead:  followed by ...
- (?!...)  negative lookahead:  NOT followed by ...
- (?<=...) positive lookbehind: preceded by ...
- (?<!...) negative lookbehind: NOT preceded by ...

In [ ]:
prices = "apples: $5, oranges: 3, pears: $12"
print("numbers preceded by $ (lookbehind):", re.findall(r"(?<=\$)\d+", prices))
print("digits followed by comma (lookahead):", re.findall(r"\d+(?=,)", prices))
print("word 'cat' NOT followed by 'nap':", re.findall(r"cat(?!nap)", "cat catnap category"))
print("word 'foo' NOT preceded by 'not':", re.findall(r"(?<!not )foo", "foo and not foo"))

## 14. Backreferences - Matching Repeated Text

- \1, \2, ... refer back to whatever an earlier group captured, letting
  you match REPEATED text (like accidental doubled words).

In [ ]:
doubled_words_text = "this this is is a a test sentence"
print("repeated words found:", re.findall(r"\b(\w+)\s+\1\b", doubled_words_text))

## 15. Named Group Backreferences

- The same idea, but referencing a NAMED group with (?P=name) instead of
  a numbered \1 - more readable in complex patterns.

In [ ]:
tag_text = "<div>content</div>"
tag_match = re.match(r"<(?P<tag>\w+)>.*</(?P=tag)>", tag_text)
print("matching open/close tag pair:", tag_match.group() if tag_match else None)
mismatched = re.match(r"<(?P<tag>\w+)>.*</(?P=tag)>", "<div>content</span>")
print("mismatched tags (no match):", mismatched)

## 16. Catastrophic Backtracking (conceptual warning)

- Nested quantifiers like (a+)+ can cause the regex engine to try an
  EXPONENTIAL number of ways to split the input among the repeated
  groups when the overall match ultimately fails. On a long enough
  input this can hang a program for minutes or longer.
- This demo does NOT actually run the slow pattern - it only explains
  why `(a+)+b` against a long run of "a"s (with no trailing "b") is
  dangerous, and shows the safe, non-backtracking equivalent.

In [ ]:
safe_pattern = re.compile(r"a+b")
print("re.match(r'a+b', 'aaab'):", bool(safe_pattern.match("aaab")))
print("re.match(r'a+b', 'aaa') (fails fast, no ambiguity to backtrack through):",
      bool(safe_pattern.match("aaa")))

## 17. re.VERBOSE - Readable Multi-line Patterns

- re.VERBOSE (or re.X) lets you spread a pattern across multiple lines
  with comments and insignificant whitespace, ignored at match time.
  Use \s or [ ] to match an actual literal space.

In [ ]:
verbose_email_pattern = re.compile(r"""
    ^                   # start of string
    [\w.\-]+            # username part
    @                   # the @ symbol
    [\w\-]+             # domain name
    \.\w+               # .com / .org / etc.
    $                   # end of string
""", re.VERBOSE)
print("verbose pattern matches 'good@example.com':", bool(verbose_email_pattern.match("good@example.com")))